# 3D TIFF Basic Porosity Analysis

3D SEM / FIB-SEMで取得した3D TIFFスタック画像から、基本的な空隙率とX/Y/Z方向の空隙率プロファイルを計算します。

このNotebookは、孔径分布、連結性、tortuosity解析の前段階として、画像の読み込み、確認、ノイズ除去、二値化、空隙率計算、結果保存を行います。


## 重要な注意

二値化条件は空隙率に大きく影響します。計算値を採用する前に、必ず中央スライス、ヒストグラム、pore maskを目視確認してください。

SEM / FIB-SEM画像では、カーテニング、チャージアップ、輝度ドリフト、再付着、視野の代表性が結果に影響します。


## 1. パラメータ設定

`INPUT_TIF`、`OUTPUT_DIR`、`PORE_IS_DARK`を必要に応じて変更してください。


In [ ]:
from pathlib import Path

# 入力3D TIFFファイル
INPUT_TIF = "data/sample_3d_sem.tif"

# 結果の出力フォルダ
OUTPUT_DIR = "outputs/porosity_output"

# 孔が暗い画像なら True、孔が明るい画像なら False
PORE_IS_DARK = True

# メディアンフィルタ設定
USE_MEDIAN_FILTER = True
MEDIAN_SIZE = 2

# 手動しきい値を使う場合は True
USE_MANUAL_THRESHOLD = False
MANUAL_THRESHOLD = 128

input_path = Path(INPUT_TIF)
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Input: {input_path}")
print(f"Output directory: {output_dir}")


## 2. ライブラリ読み込み


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from scipy import ndimage as ndi
from skimage.filters import threshold_otsu

plt.rcParams["figure.figsize"] = (6, 5)
plt.rcParams["image.cmap"] = "gray"


## 3. 3D TIFF読み込み

このNotebookでは、配列shapeを基本的に `(Z, Y, X)` と仮定します。


In [ ]:
if not input_path.exists():
    raise FileNotFoundError(
        f"Input TIFF file not found: {input_path}. "
        "Place your 3D TIFF file under data/ or update INPUT_TIF."
    )

volume = tifffile.imread(input_path)

if volume.ndim != 3:
    raise ValueError(f"Expected a 3D TIFF stack with shape (Z, Y, X), but got shape {volume.shape}")

z_size, y_size, x_size = volume.shape
print("shape (Z, Y, X):", volume.shape)
print("dtype:", volume.dtype)
print("min / max:", np.min(volume), np.max(volume))
print("mean / std:", float(np.mean(volume)), float(np.std(volume)))


## 4. 中央スライスとヒストグラムの確認


In [ ]:
center_z = z_size // 2
center_slice = volume[center_z]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(center_slice, cmap="gray")
axes[0].set_title(f"Center slice: Z={center_z}")
axes[0].axis("off")

axes[1].hist(volume.ravel(), bins=256, color="tab:blue", alpha=0.8)
axes[1].set_title("Intensity histogram")
axes[1].set_xlabel("Intensity")
axes[1].set_ylabel("Voxel count")
plt.tight_layout()
plt.show()

plt.imsave(output_dir / "center_slice_original.png", center_slice, cmap="gray")


## 5. ノイズ除去

`USE_MEDIAN_FILTER = True` の場合、3Dメディアンフィルタで孤立ノイズを低減します。


In [ ]:
if USE_MEDIAN_FILTER:
    filtered = ndi.median_filter(volume, size=MEDIAN_SIZE)
else:
    filtered = volume.copy()

filtered_center_slice = filtered[center_z]
plt.imshow(filtered_center_slice, cmap="gray")
plt.title("Center slice after filtering")
plt.axis("off")
plt.show()

plt.imsave(output_dir / "center_slice_filtered.png", filtered_center_slice, cmap="gray")


## 6. 二値化 / segmentation

デフォルトではOtsu法でしきい値を決定します。必要に応じて手動しきい値に切り替えてください。


In [ ]:
if USE_MANUAL_THRESHOLD:
    threshold = MANUAL_THRESHOLD
else:
    threshold = threshold_otsu(filtered)

if PORE_IS_DARK:
    pore_mask = filtered < threshold
else:
    pore_mask = filtered > threshold

solid_mask = ~pore_mask

print("threshold:", threshold)
print("pore voxels:", int(np.sum(pore_mask)))
print("solid voxels:", int(np.sum(solid_mask)))


## 7. pore maskの目視確認


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(center_slice, cmap="gray")
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(filtered_center_slice, cmap="gray")
axes[1].set_title("Filtered")
axes[1].axis("off")

axes[2].imshow(pore_mask[center_z], cmap="gray")
axes[2].set_title("Pore mask")
axes[2].axis("off")

plt.tight_layout()
plt.show()

plt.imsave(output_dir / "center_slice_pore_mask.png", pore_mask[center_z], cmap="gray")


## 8. 全体空隙率の計算


In [ ]:
porosity = float(np.mean(pore_mask))
solid_fraction = 1.0 - porosity

summary = pd.DataFrame([
    {
        "input_file": str(input_path),
        "shape_z": z_size,
        "shape_y": y_size,
        "shape_x": x_size,
        "dtype": str(volume.dtype),
        "threshold": float(threshold),
        "pore_is_dark": PORE_IS_DARK,
        "porosity": porosity,
        "porosity_percent": porosity * 100.0,
        "solid_fraction": solid_fraction,
        "solid_fraction_percent": solid_fraction * 100.0,
    }
])

summary.to_csv(output_dir / "porosity_summary.csv", index=False)
summary


## 9. X/Y/Z方向の空隙率プロファイル


In [ ]:
porosity_z = pore_mask.mean(axis=(1, 2))
porosity_y = pore_mask.mean(axis=(0, 2))
porosity_x = pore_mask.mean(axis=(0, 1))

pd.DataFrame({"z_index": np.arange(z_size), "porosity": porosity_z}).to_csv(output_dir / "porosity_z_profile.csv", index=False)
pd.DataFrame({"y_index": np.arange(y_size), "porosity": porosity_y}).to_csv(output_dir / "porosity_y_profile.csv", index=False)
pd.DataFrame({"x_index": np.arange(x_size), "porosity": porosity_x}).to_csv(output_dir / "porosity_x_profile.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.arange(z_size), porosity_z, marker="o", linewidth=1)
ax.set_xlabel("Z slice index")
ax.set_ylabel("Porosity")
ax.set_title("Z-direction porosity profile")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / "porosity_z_profile.png", dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.arange(x_size), porosity_x, label="X profile", linewidth=1)
ax.plot(np.arange(y_size), porosity_y, label="Y profile", linewidth=1)
ax.set_xlabel("Pixel index")
ax.set_ylabel("Porosity")
ax.set_title("X/Y-direction porosity profiles")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / "porosity_xy_profile.png", dpi=200)
plt.show()


## 10. 結果の確認

出力フォルダに画像とCSVが保存されます。空隙率の数値だけでなく、必ずpore mask画像と方向別プロファイルも確認してください。


In [ ]:
print("Saved files:")
for path in sorted(output_dir.glob("*")):
    print("-", path)
